# 1 — Stage 3 New setup and preflight
Verify pinned code, environments, assets, authentication, tests, and one idle A100. Writes `~/stage3_new/stage3_new_preflight_environment.json`.


In [ ]:
import json, os, platform, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"
P=Path.home()/"LIBERO-plus"; NATIVE=Path.home()/"stage1-native"; OUT=Path.home()/"stage3_new"
for path in (R,ID,OOD,P/"libero/libero/assets",NATIVE/"lib"):
    if not path.exists(): raise SystemExit(f"STOP: missing {path}; complete Stage 1 setup first")
OUT.mkdir(exist_ok=True)
test_env=os.environ.copy(); test_env.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
tests=subprocess.run([str(ID),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=test_env,text=True,capture_output=True); print(tests.stdout,tests.stderr); tests.check_returncode()
gpu="1"; query="name,memory.total,memory.used,utilization.gpu,driver_version"
line=subprocess.run(["nvidia-smi",f"--id={gpu}",f"--query-gpu={query}","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print("GPU",gpu,line)
name,total,used,util,driver=[x.strip() for x in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit("STOP: choose an idle physical A100 and update gpu")
(Path.home()/"stage3_new_gpu.txt").write_text(gpu)
bench=subprocess.run(["git","-C",str(R),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
pkg='import importlib.metadata,json,platform; n=("torch","lerobot","mujoco","robosuite","libero","transformers"); print(json.dumps({"python":platform.python_version(),"packages":{x:importlib.metadata.version(x) for x in n}}))'
def info(py,env=None): return json.loads(subprocess.run([str(py),"-c",pkg],env=env,capture_output=True,text=True,check=True).stdout)
provenance={"validation_passed":True,"repository_sha":bench,"libero_plus_sha":plus,"lerobot_revision":"2aba372b4e217cc47db28e0f836859b20d1456c9","model_revision":"8e174154ef5f6c60a8da12ae99c303d8963138c1","gpu":{"physical_id":gpu,"name":name,"memory_total_mib":int(total),"driver_version":driver},"host":platform.platform(),"id_environment":info(ID,test_env),"ood_environment":info(OOD,{**test_env,"PYTHONPATH":str(P)})}
(OUT/"stage3_new_preflight_environment.json").write_text(json.dumps(provenance,indent=2,sort_keys=True)+"\n")
print("PASS: Stage 3 New preflight complete; provenance saved")
